# Checkpoints, Replay and What Exactly Once Costs

Companion notebook for the [Exactly Once in Stream Pipelines](https://codemia.io/courses/big_data_engines/bd_exactly_once_in_stream_pipelines) lesson on Codemia.

**What you will build**: a simulator for a checkpointing stream job that fails every few hours. You will find the checkpoint
interval that keeps the pipeline fresh, watch the same dial decide how many duplicate rows reach the
warehouse, price the three sink strategies against each other, and work out why a long interval and a
transactional sink are a configuration that cannot run.

## Setup

In [ ]:
%pip install -q numpy matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

RATE = 180_000          # events per second arriving on the topic
CAPACITY = 240_000      # events per second the job can process, so 33% headroom
ALIGN_S = 8.0           # barrier alignment: processing stops while the snapshot is taken
STATE_GB = 40.0         # keyed state across all operators
UPLOAD_GB_S = 0.6       # snapshot upload to object storage
DOWNLOAD_GB_S = 0.8     # state restore on recovery
RESTART_S = 45.0        # pod scheduling, image pull, topic rebalance
CHANGE_GB_S = 0.05      # how fast the state is dirtied, so how big an incremental delta is
MTBF_S = 6 * 3600       # one failure every six hours: spot reclaims, OOMs, node upgrades

DRAIN = (CAPACITY - RATE) / RATE     # seconds of backlog cleared per second of processing
DOWNTIME_S = RESTART_S + STATE_GB / DOWNLOAD_GB_S

print(f'arrival rate      {RATE:>9,} events/s')
print(f'capacity          {CAPACITY:>9,} events/s')
print(f'headroom          {CAPACITY / RATE - 1:>9.0%}, so one second of backlog takes '
      f'{1 / DRAIN:.1f} s to clear')
print(f'state             {STATE_GB:>9,.0f} GB')
print(f'downtime per failure {DOWNTIME_S:>6.0f} s '
      f'({RESTART_S:.0f} s restart + {STATE_GB / DOWNLOAD_GB_S:.0f} s state restore)')
print(f'failures          {86_400 / MTBF_S:>9.0f} per day on average')

## Concept recap

A stream job has no end, so it cannot be restarted from the beginning. Instead it periodically writes
a consistent snapshot of all of its state, together with the offsets it had consumed at that moment,
and a failure means going back to the most recent snapshot and reading the log forward again from
those offsets.

Two consequences follow, and they are the whole subject. First, everything between the last
checkpoint and the failure is processed twice, so the effects of that work reach the outside world
twice unless the sink is built to prevent it. Exactly once never means the processing happens once.
It means the effect happens once, which is a property of the sink and not of the engine.

Second, the checkpoint interval is a single dial with two opposed costs. Checkpointing often stalls
the pipeline often. Checkpointing rarely makes every failure expensive. Everything below is an
attempt to put numbers on that trade with a particular job's constants rather than reasoning about it
in the abstract.

## What a snapshot costs before anything fails

In [ ]:
def snapshot_s(interval, incremental=True):
    gb = min(STATE_GB, interval * CHANGE_GB_S) if incremental else STATE_GB
    return gb / UPLOAD_GB_S


print(f'{"interval":>10}{"full snapshot":>16}{"incremental":>14}'
      f'{"full fits?":>13}{"stall share":>14}{"usable capacity":>18}')
for interval in [10, 30, 60, 120, 300, 600, 1_800]:
    full, inc = snapshot_s(interval, False), snapshot_s(interval)
    stall = ALIGN_S / interval
    usable = CAPACITY * (1 - stall)
    fits = 'yes' if full < interval else 'no'
    print(f'{interval:>8} s{full:>14.0f} s{inc:>12.1f} s{fits:>13}'
          f'{stall:>13.1%}{usable:>15,.0f}/s')

floor_s = ALIGN_S / (1 - RATE / CAPACITY)
print()
print(f'the arithmetic floor on the interval: alignment costs {ALIGN_S:.0f} s of arrivals, '
      f'and only {DRAIN:.2f} s')
print(f'of backlog clears per second, so the interval must exceed '
      f'{floor_s:.0f} s or the job never catches up.')
print(f'full snapshots of {STATE_GB:.0f} GB take {snapshot_s(0, False):.0f} s, which rules out '
      f'every interval below that.')

Two independent floors on the interval, before a single failure has happened. The alignment stall is
the sharper one: every checkpoint stops processing for eight seconds, eight seconds of arrivals pile
up, and the job only clears a third of a second of backlog per second of work. Below about thirty two
seconds the backlog from one checkpoint has not cleared when the next one starts, and the lag grows
without bound while every component reports itself healthy.

The second floor is the snapshot itself. A full snapshot of forty gigabytes takes over a minute to
upload, so a job with full snapshots cannot checkpoint faster than that, which is the practical reason
incremental snapshots exist rather than a performance nicety. With incremental snapshots the upload
is proportional to how much state was dirtied, which at these constants is seconds.

## A job that fails every six hours

In [ ]:
def time_above(segs, threshold):
    """Seconds spent above a lag threshold, over a list of linear (duration, from, to) ramps."""
    total = 0.0
    for d, a, b in segs:
        if a <= threshold and b <= threshold:
            continue
        if a >= threshold and b >= threshold:
            total += d
        else:
            total += d * (max(a, b) - threshold) / abs(a - b)
    return total


def run(interval, days=14, mtbf_s=MTBF_S, align_s=ALIGN_S, downtime_s=DOWNTIME_S,
        slo_s=60.0, seed=0):
    """Simulate checkpoints and failures, tracking how far behind the job is.

    lag is measured in seconds of event time. Alignment adds to it, processing drains
    it, and a failure adds everything since the last checkpoint plus the downtime.
    """
    rng = np.random.default_rng(seed)
    horizon = days * 86_400
    fails, t = [], 0.0
    while True:
        t += rng.exponential(mtbf_s)
        if t > horizon:
            break
        fails.append(t)

    lag = 0.0
    now = 0.0
    last_ckpt_t, lag_at_ckpt = 0.0, 0.0
    process_s = max(1.0, interval - align_s)     # the interval includes its own stall
    next_ckpt = process_s
    segs, replays = [], []
    stalled = 0.0
    fi = 0

    while now < horizon:
        next_fail = fails[fi] if fi < len(fails) else float('inf')
        t_next = min(next_ckpt, next_fail, horizon)
        d = t_next - now
        if d > 0:                                  # processing, so the lag drains
            if lag <= 0:
                segs.append((d, 0.0, 0.0))
                lag = 0.0
            elif DRAIN * d >= lag:
                t_zero = lag / DRAIN
                segs.append((t_zero, lag, 0.0))
                segs.append((d - t_zero, 0.0, 0.0))
                lag = 0.0
            else:
                nxt = lag - DRAIN * d
                segs.append((d, lag, nxt))
                lag = nxt
            now = t_next
        if now >= horizon:
            break
        if next_fail <= next_ckpt:                 # crash, restore, replay
            # the restored state is as of the checkpoint's position in the log
            replay = (now - last_ckpt_t) + lag_at_ckpt
            replays.append(max(0.0, replay - lag))  # the part that is processed twice
            new_lag = replay + downtime_s
            segs.append((downtime_s, lag, new_lag))
            lag = new_lag
            now += downtime_s
            fi += 1
            next_ckpt = now + process_s            # the old checkpoint still stands
        else:                                      # checkpoint, so processing stalls
            segs.append((align_s, lag, lag + align_s))
            lag += align_s
            now += align_s
            stalled += align_s
            last_ckpt_t, lag_at_ckpt = now, lag
            next_ckpt = now + process_s

    total = sum(d for d, _, _ in segs)
    mean_lag = sum(d * (a + b) / 2 for d, a, b in segs) / total
    max_lag = max(max(a, b) for _, a, b in segs)
    replayed_s = float(np.sum(replays))
    return {'interval': interval, 'mean_lag': mean_lag, 'max_lag': max_lag,
            'end_lag': lag, 'failures': len(fails),
            'stall_share': stalled / total,
            'mins_over_slo': time_above(segs, slo_s) / 60 / days,
            'replay_s_per_fail': float(np.mean(replays)) if replays else 0.0,
            'dups_per_day': replayed_s * RATE / days,
            'lost_s_per_day': (stalled + replayed_s) / days,
            'segs': segs}


for interval in [30, 300]:
    r = run(interval)
    print(f'interval {interval:>5} s: mean lag {r["mean_lag"]:>8.1f} s, '
          f'max lag {r["max_lag"]:>8.0f} s, lag at the end of two weeks '
          f'{r["end_lag"]:>8.0f} s')

The thirty second configuration is the failure mode the floor predicted, seen from the other end: it
is below thirty two seconds, so the lag at the end of the two week run is enormous and still growing.
Nothing crashed. The job is simply spending more time stalled for checkpoints than it can make up,
and it will keep falling behind for as long as it runs.

The five minute configuration is stable, and its mean lag is small. The maximum tells you where the
real cost is, and that is what the sweep below is for.

In [ ]:
print(f'{"interval":>10}{"stall":>9}{"mean lag":>11}{"max lag":>10}'
      f'{"replay/failure":>17}{"duplicates/day":>17}{"min/day over SLO":>19}')
rows = []
for interval in [30, 40, 60, 90, 120, 300, 600, 1_200, 1_800, 3_600]:
    runs = [run(interval, seed=s) for s in range(3)]
    r = {k: float(np.mean([x[k] for x in runs]))
         for k in ['stall_share', 'mean_lag', 'max_lag', 'replay_s_per_fail',
                   'dups_per_day', 'mins_over_slo', 'lost_s_per_day', 'end_lag']}
    r['interval'] = interval
    rows.append(r)
    runaway = (f' <- still {r["end_lag"] / 60:.0f} min behind after two weeks'
               if r['end_lag'] > 600 else '')
    print(f'{interval:>8} s{r["stall_share"]:>9.1%}{r["mean_lag"]:>10.1f} s'
          f'{r["max_lag"]:>9.0f} s{r["replay_s_per_fail"]:>15.0f} s'
          f'{r["dups_per_day"] / 1e6:>14.1f} M{r["mins_over_slo"]:>17.0f}{runaway}')

stable = [r for r in rows if r['end_lag'] < 600]
best_fresh = min(stable, key=lambda r: r['mins_over_slo'])
best_waste = min(stable, key=lambda r: r['lost_s_per_day'])
print()
print(f'freshest: {best_fresh["interval"]:.0f} s interval, '
      f'{best_fresh["mins_over_slo"]:.0f} minutes a day above a 60 s lag objective')
print(f'cheapest: {best_waste["interval"]:.0f} s interval, '
      f'{best_waste["lost_s_per_day"]:.0f} s a day of work stalled or redone')
print(f'at the freshest setting the sink still receives '
      f'{best_fresh["dups_per_day"] / 1e6:.1f} M duplicate events a day')

The sweep has a minimum rather than a direction, which is the point. Short intervals stall constantly
and shrink the headroom that recovery depends on, so a failure takes longer to recover from even
though it replays less. Long intervals barely stall but replay ten to fifteen minutes of events, and
those replays are what put the job above its freshness objective for hours a day.

Notice that the freshest setting and the cheapest setting are not the same number. Minimising wasted
compute pushes the interval out, because a stall happens on a schedule and a failure only happens
every six hours. Minimising time spent behind pulls it in. Which one you optimise is a product
decision, and it is worth knowing that you are making it.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.8))

for interval, colour in [(90, 'tab:green'), (1_200, 'tab:red')]:
    segs = run(interval, days=2, seed=4)['segs']
    t, lag = [0.0], [0.0]
    for d, a, b in segs:
        t.append(t[-1] + d)
        lag.append(b)
    ax1.plot(np.array(t) / 3600, lag, color=colour, lw=1.0,
             label=f'{interval} s interval')
ax1.axhline(60, ls=':', color='black', label='60 s freshness objective')
ax1.set_xlabel('hours')
ax1.set_ylabel('lag (seconds of event time)')
ax1.set_yscale('symlog', linthresh=10)
ax1.set_title('Two days of the same job at two intervals')
ax1.legend()
ax1.grid(alpha=0.3)

iv = np.array([r['interval'] for r in rows])
ok = np.array([r['end_lag'] < 600 for r in rows])
ax2.plot(iv[ok], [r['mins_over_slo'] for r, k in zip(rows, ok) if k], 'o-',
         color='tab:blue', label='minutes a day above the objective')
ax2.set_xscale('log')
ax2.set_xlabel('checkpoint interval (s)')
ax2.set_ylabel('minutes a day over the objective', color='tab:blue')
ax2.grid(alpha=0.3)
ax2b = ax2.twinx()
ax2b.plot(iv[ok], [r['dups_per_day'] / 1e6 for r, k in zip(rows, ok) if k], 's--',
          color='tab:orange')
ax2b.set_ylabel('duplicate events per day (millions)', color='tab:orange')
ax2.set_title('Freshness has an optimum, duplicates do not')
plt.show()

The left panel is the shape worth remembering. The short interval keeps a sawtooth of small spikes
and recovers from each failure quickly. The long interval is flat and clean for twenty minutes at a
time, then jumps to twenty minutes of backlog and spends a long while grinding it down, because
clearing backlog is three times slower than accumulating it at this headroom.

The right panel is the asymmetry that decides the setting. Freshness has an optimum, so there is a
best interval to pick. Duplicates do not: every increase in the interval sends more duplicate events
to the sink, with no minimum anywhere. If the sink cannot absorb duplicates, the interval is not a
tuning knob, it is a correctness parameter.

## The classic formula, and what it does not know

In [ ]:
daly = np.sqrt(2 * ALIGN_S * MTBF_S)
print(f'the textbook optimum, sqrt(2 * checkpoint cost * MTBF): {daly:.0f} s')
print(f'measured minimum for wasted work:                      '
      f'{best_waste["interval"]:.0f} s')
print(f'measured minimum for time spent behind:                '
      f'{best_fresh["interval"]:.0f} s')
print()
print('what the formula leaves out, with this job\'s numbers:')
print(f'  recovery is not instant: {DOWNTIME_S:.0f} s of downtime per failure, '
      f'{DOWNTIME_S * 86_400 / MTBF_S / 60:.0f} min/day')
print(f'  backlog clears {1 / DRAIN:.1f}x slower than it accumulates, so a '
      f'{daly / 60:.0f} min replay costs {daly / DRAIN / 60:.0f} min of catching up')
print(f'  the stall itself eats headroom, which is why intervals under '
      f'{floor_s:.0f} s diverge')

The square root formula is the right instinct and it is derived from exactly two quantities, the cost
of a checkpoint and the mean time between failures. It lands in the same neighbourhood as the
measured minimum for wasted work, which is the thing it is actually optimising.

It does not know about the three effects printed underneath, and those are what make the freshness
optimum much tighter than the waste optimum. Recovery has a fixed cost that the interval cannot
reduce. Backlog clears more slowly than it builds, so a replay of a given length costs several times
its own length in catch up time. And the stall consumes the very headroom that catching up depends
on. Use the formula to find the neighbourhood, then measure.

## What the sink does with the duplicates

In [ ]:
DAILY_EVENTS = RATE * 86_400
DEDUP_WINDOW_S = 300
KEY_BYTES = 40


def sink_report(r):
    dups = r['dups_per_day']
    replay = r['replay_s_per_fail']
    windows = replay / 60 * (86_400 / MTBF_S)      # one minute windows double counted per day
    return {'dups': dups, 'share': dups / DAILY_EVENTS, 'windows': windows,
            'visibility': r['interval'] / 2}


print('a sink with no protection, at three intervals:')
print(f'{"interval":>10}{"duplicates/day":>17}{"share of volume":>18}'
      f'{"1-min windows double counted/day":>36}')
for interval in [90, 300, 1_200]:
    r = next(x for x in rows if x['interval'] == interval)
    rep = sink_report(r)
    print(f'{interval:>8} s{rep["dups"] / 1e6:>15.1f} M{rep["share"]:>18.3%}'
          f'{rep["windows"]:>36.0f}')

print()
print('the share of daily volume looks harmless. the per window error does not:')
r = next(x for x in rows if x['interval'] == 300)
print(f'  at a 300 s interval a failure replays {r["replay_s_per_fail"]:.0f} s of events,')
print(f'  so every one minute window inside that span is reported at 200% of the truth,')
print(f'  and a dashboard averaging the day shows an error of '
      f'{sink_report(r)["share"]:.3%}.')
print('  the daily number is defensible. the alert that fired on one window was not.')

This is why duplicate tolerance is discussed per query rather than per pipeline. Ninety five million
duplicate events a day sounds alarming and is six thousandths of the volume, which no business report
would notice. The same duplicates mean that for a few minutes after every failure, a one minute window is counted
twice, so a count or a sum over that window is double, and anything watching those windows, an alert
threshold, an anomaly detector, a page that says traffic doubled, is wrong in a way that looks
exactly like a real event.

Deduplication and transactions both exist to remove that, and they have different prices.

In [ ]:
print('three ways to make the effect happen once, at a 300 s interval:')
r = next(x for x in rows if x['interval'] == 300)
dups = r['dups_per_day']
upsert_amp = dups / DAILY_EVENTS

print()
print('1. at least once, no protection')
print(f'   correct?            no, {dups / 1e6:.1f} M duplicate rows a day')
print(f'   added latency       none')
print(f'   added cost          none')
print()
print('2. idempotent writes, upsert keyed on a deterministic event id')
print(f'   correct?            yes, for counts and last-value state. '
      f'not for additive counters')
print(f'   added latency       none')
print(f'   added cost          {upsert_amp:.3%} extra writes, and every write becomes '
      f'an upsert')
print(f'   requires            a key derived from the event, never from processing time '
      f'or a uuid')
print()
print('3. transactional sink, output committed with the checkpoint')
print(f'   correct?            yes, including additive counters')
print(f'   added latency       {r["interval"] / 2:.0f} s average, '
      f'{r["interval"]:.0f} s worst case: nothing is visible until the commit')
print(f'   added cost          one transaction per partition per checkpoint')
print()
dedup_keys = RATE * DEDUP_WINDOW_S
print('and a fourth that people reach for first, an explicit dedup window:')
print(f'   {DEDUP_WINDOW_S} s of keys at {RATE:,}/s is {dedup_keys / 1e6:.0f} M keys, '
      f'{dedup_keys * KEY_BYTES / 1e9:.1f} GB of extra state')
print(f'   which is {dedup_keys * KEY_BYTES / 1e9 / STATE_GB:.0%} of the state you already '
      f'had, so snapshots and restores')
print(f'   get that much slower, and a replay longer than {DEDUP_WINDOW_S} s '
      f'slips past it anyway')
print(f'   at a 1200 s interval the mean replay is '
      f'{next(x for x in rows if x["interval"] == 1_200)["replay_s_per_fail"]:.0f} s, '
      f'which is exactly that case')

The ranking is not absolute, it depends on what the sink stores. An upsert keyed on an event id is
free and sufficient when the destination holds one row per event or the latest value per key, and it
is useless for a running total, because adding the same amount twice is not fixed by writing it
twice. A transactional sink fixes the running total and charges for it in freshness, since nothing is
readable until the checkpoint commits.

The dedup window is the option worth being suspicious of. It looks like a local fix, it adds
gigabytes to the state that every snapshot and every restore has to carry, and it is sized in exactly
the units, seconds of replay, that a long checkpoint interval exceeds. The cases it misses are the
ones caused by the setting you chose to make it necessary.

In [ ]:
TXN_TIMEOUT_DEFAULT_S = 60.0       # Kafka transaction.timeout.ms default
TXN_TIMEOUT_MAX_S = 900.0          # transaction.max.timeout.ms broker side default

print('a transactional sink needs the transaction to outlive the checkpoint interval')
print('plus the time a recovery takes to decide the transaction\'s fate:')
print()
print(f'{"interval":>10}{"needs at least":>18}{"60 s default":>15}{"900 s max":>12}')
for interval in [60, 90, 300, 600, 1_200, 1_800]:
    needed = interval + DOWNTIME_S
    print(f'{interval:>8} s{needed:>16.0f} s'
          f'{"ok" if needed <= TXN_TIMEOUT_DEFAULT_S else "times out":>15}'
          f'{"ok" if needed <= TXN_TIMEOUT_MAX_S else "impossible":>12}')

print()
print(f'the {TXN_TIMEOUT_DEFAULT_S:.0f} s default supports no interval at all on this job:')
print(f'deciding the fate of a transaction waits for the recovery, and that alone '
      f'takes {DOWNTIME_S:.0f} s.')
print()
print('a timed out transaction is not a delay. the broker aborts it, the job restores')
print('from a checkpoint whose output was discarded, and the pipeline either loses that')
print('window or refuses to start. so the interval and the sink are one decision:')
print(f'  with a transactional sink the interval must satisfy '
      f'{floor_s:.0f} s < interval < '
      f'{TXN_TIMEOUT_MAX_S - DOWNTIME_S:.0f} s')
print(f'  the freshness optimum, {best_fresh["interval"]:.0f} s, sits inside that window')
print(f'  the waste optimum, {best_waste["interval"]:.0f} s, '
      f'{"sits inside it too" if best_waste["interval"] < TXN_TIMEOUT_MAX_S - DOWNTIME_S else "does not"}')

This is the constraint that turns a tuning exercise into a design one. The interval has a lower bound
from alignment and headroom, and an upper bound from the sink's transaction lifetime, and both bounds
were computed from numbers that belong to this job rather than from advice. Any interval outside that
band is not slow or wasteful, it does not run.

Notice that the default transaction timeout supports no interval on this job at all. Recovery takes
ninety five seconds on its own, so a sixty second transaction has expired before anyone can decide
whether to commit it, and a team enabling exactly once has to raise that timeout deliberately before
the pipeline will survive its first failure.

It is also why raising the checkpoint interval later, to reduce overhead, can break exactly once
without anybody touching the sink configuration. The two settings are coupled through a timeout that
nobody changed.

## What to take away

- Exactly once is not a property of the engine. Processing happens more than once by design, and only
  the sink can make the effect happen once.
- The checkpoint interval has two hard floors: the alignment stall must fit inside the headroom, and a
  full snapshot must fit inside the interval. Below either one the job falls behind forever while
  reporting itself healthy.
- Freshness has an optimum interval, waste has a different one, and duplicates have none. Decide
  which of the three you are optimising before picking a number.
- Backlog clears more slowly than it accumulates, by the inverse of the headroom. A replay of fifteen
  minutes at thirty three percent headroom costs three quarters of an hour of catching up.
- Duplicates are a small share of daily volume and a total distortion of individual windows. Judge
  them per query, not per pipeline.
- Idempotent upserts are free and insufficient for additive state. Transactional sinks are correct and
  charge the interval in latency. Dedup windows add state in exactly the dimension that a long
  interval overruns.
- A transactional sink bounds the interval from above through its transaction timeout, so the interval
  and the sink are one decision rather than two.